In [1]:
# Emerging Technologies — Problems

In [2]:
## Problem 1: Generating Random Boolean Functions

This problem asks for a Python function `random_constant_balanced` that returns a
randomly chosen function from the set of **constant** or **balanced** Boolean
functions taking four Boolean arguments as input.

### Background

The Deutsch–Jozsa algorithm [1] is one of the earliest examples of a quantum
algorithm that provides a provable speedup over deterministic classical
algorithms. It is designed to decide, with a single query, whether a given
Boolean function $f : \{0,1\}^n \to \{0,1\}$ belongs to one of two restricted
classes:

- **Constant functions** — $f(x) = c$ for every input $x$, where $c \in \{0,1\}$.
- **Balanced functions** — $f$ returns $0$ on exactly half of the $2^n$ possible
  inputs and $1$ on the other half.

A general Boolean function need not be either of these; in fact, most are
neither. The Deutsch–Jozsa problem is restricted *by promise* to only these two
classes. The modern textbook formulation of the algorithm given by Cleve,
Ekert, Macchiavello and Mosca [2] is the one used in most quantum-computing
tutorials today, including the IBM Quantum Learning material [3] linked in the
problem statement.

Before we can simulate or analyse the algorithm in later problems, we need a
way to generate such functions at random — that is the goal of Problem 1.

### References

[1] D. Deutsch and R. Jozsa, "Rapid solution of problems by quantum
computation," *Proceedings of the Royal Society A*, vol. 439, no. 1907,
pp. 553–558, 1992. https://doi.org/10.1098/rspa.1992.0167

[2] R. Cleve, A. Ekert, C. Macchiavello, and M. Mosca, "Quantum algorithms
revisited," *Proceedings of the Royal Society A*, vol. 454, no. 1969,
pp. 339–354, 1998. https://doi.org/10.1098/rspa.1998.0164

[3] IBM Quantum Learning, "The Deutsch–Jozsa algorithm."
https://quantum.cloud.ibm.com/learning/en/modules/computer-science/deutsch-jozsa

SyntaxError: invalid character '–' (U+2013) (2493897688.py, line 9)

In [5]:
### Counting constant and balanced functions on four inputs

A Boolean function $f : \{0,1\}^4 \to \{0,1\}$ is fully specified by its
**truth table** — the list of outputs for each of the $2^4 = 16$ possible
input combinations. Since each of those 16 outputs is independently $0$ or
$1$, there are $2^{16} = 65{,}536$ Boolean functions on four inputs in total
[1].

We are interested in two specific subsets:

**Constant functions.** There are exactly **two**: the function that returns
$0$ everywhere, and the function that returns $1$ everywhere.

**Balanced functions.** A balanced function returns $1$ on exactly half of
the $2^4 = 16$ inputs, and $0$ on the other half. The number of such
functions is therefore the number of ways to choose which 8 of the 16 input
combinations map to $1$:

$$
\binom{16}{8} = 12{,}870.
$$

So the *promise set* for the four-input Deutsch–Jozsa problem contains
$2 + 12{,}870 = 12{,}872$ functions in total. Out of all $65{,}536$ Boolean
functions on four inputs, only about **19.6%** satisfy the Deutsch–Jozsa
promise — the rest are neither constant nor balanced and the algorithm is
not defined for them [2].

This count drives the implementation strategy: rather than rejection-sampling
from all $2^{16}$ functions (which would discard roughly four out of every
five candidates), we will sample directly from the constant and balanced
classes in proportion to their sizes.

### References

[1] D. E. Knuth, *The Art of Computer Programming, Volume 4A: Combinatorial
Algorithms, Part 1*. Upper Saddle River, NJ: Addison-Wesley, 2011, ch. 7.1.1.

[2] M. A. Nielsen and I. L. Chuang, *Quantum Computation and Quantum
Information*, 10th anniversary ed. Cambridge: Cambridge University Press,
2010, sec. 1.4.3.

SyntaxError: invalid syntax (3915817419.py, line 3)

In [6]:
### Representation strategy: truth tables as integers

A Boolean function on four inputs is fully described by its 16-entry truth
table. There are several reasonable ways to represent such a table in
Python, but the most compact is a single 16-bit integer where bit $i$
holds the output $f(x)$ for the input $x$ whose binary encoding is $i$.

For example, the integer `0b1010101010101010` (which is `0xAAAA`, or
$43{,}690$ in decimal) encodes the function

$$
f(x_3, x_2, x_1, x_0) = x_0,
$$

because its truth table outputs $1$ exactly when the least significant
input bit is $1$. This is a balanced function: half of the 16 inputs have
$x_0 = 1$.

This representation has three advantages relevant to our task [1]:

1. **Constant functions are trivial to construct.** The all-zeros function
   is the integer $0$ and the all-ones function is $2^{16} - 1 = 65{,}535$.
2. **Balancedness is a single popcount.** A function is balanced on four
   inputs if and only if its 16-bit truth table has exactly 8 bits set,
   which Python exposes directly as `int.bit_count()` (added in Python 3.10).
3. **Sampling a balanced function reduces to a random bit selection.** We
   can pick which 8 of the 16 input positions map to $1$ uniformly at
   random using `random.sample`, then assemble the integer.

The actual *callable* the user receives will wrap this integer in a closure
that takes four Boolean arguments, packs them into the index $i$, and
returns the corresponding bit of the truth table.

### References

[1] H. S. Warren Jr., *Hacker's Delight*, 2nd ed. Upper Saddle River, NJ:
Addison-Wesley, 2013, ch. 5 ("Counting Bits").

SyntaxError: unterminated string literal (detected at line 36) (1074967894.py, line 36)

In [8]:
"""Imports and helpers for Problem 1."""

import random
from typing import Callable

# A Boolean function on four inputs maps four bools to a single bool.
BoolFunc4 = Callable[[bool, bool, bool, bool], bool]


def _truth_table_to_callable(table: int) -> BoolFunc4:
    """Wrap a 16-bit truth table integer in a 4-argument Boolean callable.

    Bit ``i`` of ``table`` is the output of the function on the input whose
    binary encoding (with ``x3`` as the most significant bit) equals ``i``.

    Parameters
    ----------
    table : int
        A non-negative integer in the range ``[0, 2**16)`` whose binary
        representation is the truth table of the function.

    Returns
    -------
    Callable[[bool, bool, bool, bool], bool]
        A function ``f(x3, x2, x1, x0)`` that returns the corresponding
        truth-table bit as a Python ``bool``.
    """
    if not 0 <= table < (1 << 16):
        raise ValueError(
            f"truth table must fit in 16 bits, got {table}"
        )

    def f(x3: bool, x2: bool, x1: bool, x0: bool) -> bool:
        # Pack the four input bits into an index in [0, 16).
        index = (int(bool(x3)) << 3) | (int(bool(x2)) << 2) \
                | (int(bool(x1)) << 1) | int(bool(x0))
        return bool((table >> index) & 1)

    return f

In [ ]:
### Implementing `random_constant_balanced`

We now have everything we need: a representation (16-bit truth table), a
wrapper that turns such a table into a callable, and the counts of each
class (2 constant functions, $\binom{16}{8} = 12{,}870$ balanced functions).

The function below samples uniformly from the union of these two sets. To
keep the sampling truly uniform across all $12{,}872$ functions we weight
the choice between the two branches by their class sizes — otherwise a
50/50 coin flip between "constant" and "balanced" would massively
over-represent the two constant functions.

Within each branch, the sampling is straightforward:

- **Constant branch.** Choose the all-zeros or all-ones truth table with
  equal probability.
- **Balanced branch.** Choose 8 of the 16 input positions uniformly at
  random using `random.sample`, then set those bits in the truth table.

In [9]:
def random_constant_balanced(rng: random.Random | None = None) -> BoolFunc4:
    """Return a uniformly random constant or balanced Boolean function on 4 inputs.

    The returned callable accepts four Boolean arguments and returns a single
    Boolean output. The function is drawn uniformly at random from the union
    of:

    - the 2 constant functions on 4 inputs (always ``False``, always ``True``), and
    - the C(16, 8) = 12,870 balanced functions on 4 inputs.

    The branch (constant vs. balanced) is selected with probability proportional
    to the size of each class, so every one of the 12,872 functions in the
    promise set is equally likely to be returned.

    Parameters
    ----------
    rng : random.Random, optional
        A random number generator. If ``None``, the module-level ``random``
        functions are used. Passing an explicit ``random.Random`` instance
        allows the caller to seed the sampling for reproducibility.

    Returns
    -------
    Callable[[bool, bool, bool, bool], bool]
        A Boolean function that is either constant or balanced.
    """
    # Use the supplied generator, or fall back to the module-level one.
    choice = rng.choice if rng is not None else random.choice
    sample = rng.sample if rng is not None else random.sample
    random_func = rng.random if rng is not None else random.random

    # There are 2 constant functions and C(16, 8) = 12,870 balanced functions.
    # Probability of the constant branch = 2 / 12,872.
    n_constant = 2
    n_balanced = 12_870
    p_constant = n_constant / (n_constant + n_balanced)

    if random_func() < p_constant:
        # Constant branch: 0x0000 (always False) or 0xFFFF (always True).
        table = choice([0x0000, 0xFFFF])
    else:
        # Balanced branch: pick 8 of the 16 input positions to map to True.
        ones_positions = sample(range(16), 8)
        table = 0
        for pos in ones_positions:
            table |= (1 << pos)

    return _truth_table_to_callable(table)